In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import pandas as pd

In [17]:
p = {
    'kks_1':  0.95,   #QoI
    'kkr_1':  0,   #QoI
    'kk_2':  0.95,    #SDHI
    'C50s_1': 0.02,    #QoI_azoxystrobin sensitive
    'C50s_2': 0.06,  #SDHI_fluxapyroxad sensitive
    'C50r_2': 1.4,
    'K':     1.0,    
    'beta':  0.2,    
    'mu':    0.1,    
    'rH':    0.1,   
    'rhor_1': 0,   #QoI
    'rhor_2': 0.1,     #SDHI
    'm':    3.2e-5  
}

C = np.array([0, 0.25, 0.5, 1, 2, 4]) #dose strategies, 1 represents baseline dose. (0.25-1 is selection window)

In [44]:
y0 = [0.95,     
      0.03,   
      0.0,
      0.0,
      0.0]   
t = np.linspace(0, 2000, 2000)

In [18]:
def Hills_function (C, kk, C50):
   return (kk * C / (C + C50))

C1, C2 = np.meshgrid(C, C)

e_1 = Hills_function(C1,p['kks_1'],p['C50s_1']) #QoI_azoxystrobin sensitive
es_2 = Hills_function(C2,p['kk_2'],p['C50s_2']) #SDHI_fluxapyroxad sensitive
er_2 = Hills_function(C2,p['kk_2'],p['C50r_2']) #SDHI_fluxapyroxad resistant

b_ab = p['beta']*(1-e_1)*(1-es_2)
b_Ab = p['beta']*(1-p['rhor_1'])*(1-es_2)
b_aB = p['beta']*(1-e_1)*(1-p['rhor_2'])*(1-er_2)  #SDHI resistance is quantitative, while QoI can be assumed to be qualitative
b_AB = p['beta']*(1-p['rhor_1'])*(1-p['rhor_2'])*(1-er_2)


df_doses = pd.DataFrame ({
    'QoI': C1.flatten(),
    'SDHI' : C2.flatten(),
    'b_ab': b_ab.flatten(),
    'b_Ab': b_Ab.flatten(),
    'b_aB': b_aB.flatten(),
    'b_AB': b_AB.flatten(),
    'e_1' : e_1.flatten(),
    'es_2' : es_2.flatten(),
    'er_2':er_2.flatten(),
})
print( 'fungicide mixtures and transmission rates')
df_doses

fungicide mixtures and transmission rates


,QoI,SDHI,b_ab,b_Ab,b_aB,b_AB,e_1,es_2,er_2
0,0.00,0.00,0.200000,0.200000,0.180000,0.180000,0.000000,0.000000,0.000000
1,0.25,0.00,0.024074,0.200000,0.021667,0.180000,0.879630,0.000000,0.000000
2,0.50,0.00,0.017308,0.200000,0.015577,0.180000,0.913462,0.000000,0.000000
3,1.00,0.00,0.013725,0.200000,0.012353,0.180000,0.931373,0.000000,0.000000
4,2.00,0.00,0.011881,0.200000,0.010693,0.180000,0.940594,0.000000,0.000000
5,4.00,0.00,0.010945,0.200000,0.009851,0.180000,0.945274,0.000000,0.000000
6,0.00,0.25,0.046774,0.046774,0.154091,0.154091,0.000000,0.766129,0.143939
7,0.25,0.25,0.005630,0.046774,0.018548,0.154091,0.879630,0.766129,0.143939
8,0.50,0.25,0.004048,0.046774,0.013335,0.154091,0.913462,0.766129,0.143939
9,1.00,0.25,0.003210,0.046774,0.010575,0.154091,0.931373,0.766129,0.143939


In [5]:
def ODEs (t,y,p,b_total):
    H, I_ab, I_Ab, I_aB, I_AB = y
    b_ab, b_Ab, b_aB, b_AB = b_total
    m = p['m']
    rH = p ['rH']
    K = p['K']
    mu = p['mu']

    dH = rH*(K - H - I_ab - I_Ab - I_aB - I_AB) - b_ab*I_ab*H - b_Ab*I_Ab*H - b_aB*I_aB*H - b_AB*I_AB*H
    dI_ab = (1-2*m)*b_ab*I_ab*H + m*b_Ab*I_Ab*H + m*b_aB*I_aB*H - mu*I_ab
    dI_Ab = m*b_ab*I_ab*H + (1-2*m)*b_Ab*I_Ab*H + m*b_AB*I_AB*H - mu*I_Ab
    dI_aB = m*b_ab*I_ab*H + (1-2*m)*b_aB*I_aB*H + m*b_AB*I_AB*H - mu*I_aB
    dI_AB = m*b_Ab*I_Ab*H + m*b_aB*I_aB*H + (1-2*m)*b_AB*I_AB*H - mu*I_AB

    return [dH, dI_ab, dI_Ab, dI_aB, dI_AB]

In [43]:
def calculate_90_threshold (sol):
    I_total = sol.y[1,:]+sol.y[2,:]+sol.y[3,:]+sol.y[4,:]
    
    proportion_I_AB = np.divide(
        sol.y[4,:],
        I_total,
        out=np.zeros_like(I_total), 
        where = (I_total) > 0)

    epidemic_threshold = 0.01 * p['K'] 
    validity = I_total > epidemic_threshold
    
    indices = np.where((proportion_I_AB >= 0.9) & validity & (sol.y[4,:] > 0))[0]

    if len(indices) > 0:
        return sol.t[indices[0]]
    else:
        return np.nan

In [42]:
results = []
def calculate_R0 (b_ab, b_Ab, b_aB, b_AB, p):
    R0_ab = b_ab * p['K'] / p['mu']
    R0_Ab = b_Ab * p['K'] / p['mu']
    R0_aB = b_aB * p['K'] / p['mu']
    R0_AB = b_AB * p['K'] / p['mu']
    return R0_ab, R0_Ab, R0_aB, R0_AB
        
    
for j in range(len(C)):
    for i in range(len(C)):
        e_1 = Hills_function(C[i], p['kks_1'], p['C50s_1'])
        es_2 = Hills_function(C[j], p['kk_2'], p['C50s_2'])
        er_2 = Hills_function(C[j], p['kk_2'], p['C50r_2'])
        b_ab = p['beta'] * (1 - e_1) * (1 - es_2)
        b_Ab = p['beta'] * (1 - p['rhor_1']) * (1 - es_2)
        b_aB = p['beta'] * (1 - e_1) * (1 - p['rhor_2']) * (1 - er_2)
        b_AB = p['beta'] * (1 - p['rhor_1']) * (1 - p['rhor_2']) * (1 - er_2)
        b_total = (b_ab, b_Ab, b_aB, b_AB)

        sol = solve_ivp(ODEs, [t[0], t[-1]], y0, t_eval=t, args=(p, b_total),
                        method='Radau',rtol=1e-10, atol=1e-12) 
        if not sol.success:
            print(f'integration failed at i={i}, j={j}')

        H_eq = sol.y[0, -1]
        I_ab_eq = sol.y[1, -1]
        I_Ab_eq = sol.y[2, -1]
        I_aB_eq = sol.y[3, -1]
        I_AB_eq = sol.y[4, -1]

        R0_ab, R0_Ab, R0_aB, R0_AB = calculate_R0(b_ab, b_Ab, b_aB, b_AB, p)
        t90 = calculate_90_threshold(sol)

        results.append({
            "QoI": C[i],
            "SDHI": C[j],
            "$H^*$": H_eq,
            "$I_{ab}^*$": I_ab_eq,
            "$I_{Ab}^*$": I_Ab_eq,
            "$I_{aB}^*$": I_aB_eq,
            "$I_{AB}^*$": I_AB_eq,
            "t90": t90,
            "$R_0{ab}": R0_ab,
            "$R_0{Ab}" : R0_Ab,
            "$R_0{aB}" : R0_aB,
            "$R_0{AB}" : R0_AB                       
        })

df_results = pd.DataFrame(results)
df_results

,QoI,SDHI,$H^*$,$I_{ab}^*$,$I_{Ab}^*$,$I_{aB}^*$,$I_{AB}^*$,t90,$R_0{ab},$R_0{Ab},$R_0{aB},$R_0{AB}
0,0.00,0.00,0.500016,1.866233e-01,6.328877e-02,5.972272e-05,2.025190e-05,NaN,2.000000,2.000000,1.800000,1.800000
1,0.25,0.00,0.500032,9.091496e-06,2.498949e-01,2.622460e-09,7.997186e-05,NaN,0.240741,2.000000,0.216667,1.800000
2,0.50,0.00,0.500032,8.754786e-06,2.498953e-01,2.524182e-09,7.997197e-05,NaN,0.173077,2.000000,0.155769,1.800000
3,1.00,0.00,0.500032,8.586431e-06,2.498954e-01,2.475073e-09,7.997202e-05,NaN,0.137255,2.000000,0.123529,1.800000
4,2.00,0.00,0.500032,8.502253e-06,2.498955e-01,2.450526e-09,7.997205e-05,NaN,0.118812,2.000000,0.106931,1.800000
5,4.00,0.00,0.500032,8.460164e-06,2.498956e-01,2.438254e-09,7.997207e-05,NaN,0.109453,2.000000,0.098507,1.800000
6,0.00,0.25,0.648988,4.032006e-06,4.031895e-06,8.775010e-02,8.774768e-02,NaN,0.467742,0.467742,1.540909,1.540909
7,0.25,0.25,0.649009,1.068252e-10,8.063388e-06,6.384223e-06,1.754810e-01,31.015508,0.056302,0.467742,0.185480,1.540909
8,0.50,0.25,0.649009,9.792761e-11,8.063399e-06,6.147779e-06,1.754812e-01,31.015508,0.040478,0.467742,0.133348,1.540909
9,1.00,0.25,0.649009,9.351994e-11,8.063404e-06,6.029556e-06,1.754814e-01,30.015008,0.032100,0.467742,0.105749,1.540909
